In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE CATALOG IF NOT EXISTS company_ro")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.gold")

# Gold Layer: Company Financial Summary

This notebook creates a fully denormalized, analytics-ready view of Romanian company financial data.

## Purpose

Combines financial facts with company and economic activity dimensions to create a single wide table for:
* Business intelligence dashboards
* Ad-hoc analytics queries
* Data exports for reporting
* Machine learning feature engineering

## Data Sources

* **Silver Fact Table**: `company_ro.silver.fact_financiar_anual` - Annual financial statements (15M+ rows)
* **Company Dimension**: `company_ro.silver.dim_firma` - Company master data (4M+ companies)
* **CAEN Dimension**: `company_ro.silver.dim_caen` - Economic activity classifications

## Key Features

### CAEN Code Normalization
The gold layer handles mismatches between Ministry of Finance CAEN codes and ONRC CAEN codes:
* Normalizes both to 4-digit format using `LPAD(cod, 4, '0')`
* Falls back to mfinante code if ONRC lookup fails
* Provides human-readable `caen_display_name` column

### Enrichment Columns
* `denumire` - Company name
* `judet`, `localitate`, `adresa` - Geographic location
* `stare_firma`, `is_activa` - Company status
* `forma_juridica` - Legal form (SRL, SA, PFA, etc.)
* `denumire_caen`, `grupa_caen`, `clasa_caen` - Economic activity hierarchy

### Financial Metrics (All in RON)
* Revenue: `cifra_afaceri`, `venituri_totale`
* Profitability: `profit_net`, `pierdere_neta`, `profit_brut`, `pierdere_bruta`
* Assets: `active_imobilizate`, `active_circulante`, `stocuri`, `creante`, `casa_si_conturi`
* Liabilities: `datorii`, `capitaluri_proprii`, `capital_social`
* Operations: `nr_mediu_salariati` (employees)

## Usage Patterns

```sql
-- Year-over-year revenue growth by industry
SELECT
  grupa_caen,
  an,
  SUM(cifra_afaceri) AS total_revenue
FROM company_ro.gold.company_financial_summary
WHERE an >= 2020
GROUP BY grupa_caen, an
ORDER BY an DESC, total_revenue DESC

-- Top companies by revenue in a specific region
SELECT
  denumire,
  judet,
  cifra_afaceri,
  nr_mediu_salariati
FROM company_ro.gold.company_financial_summary
WHERE an = 2024
  AND judet = 'BUCUREŞTI'
ORDER BY cifra_afaceri DESC
LIMIT 100
```

## Data Quality

* approximately 99.5% of financial records join to company dimension
* approximately 90% of CAEN codes resolve to descriptions
* Grain: One row per company per year
* Time span: 2011-2025 (15 years)
* Total records: 9M+ rows

In [0]:
display(spark.sql("SHOW TABLES IN company_ro.silver"))

In [0]:
dim_firma = spark.table("company_ro.silver.dim_firma")
dim_caen = spark.table("company_ro.silver.dim_caen")
fact_financiar = spark.table("company_ro.silver.fact_financiar_anual")

print("dim_firma columns:")
print(dim_firma.columns)

print("dim_caen columns:")
print(dim_caen.columns)

print("fact_financiar columns:")
print(fact_financiar.columns)

In [0]:
company_financial_summary = spark.sql("""
WITH fact_normalized AS (
  SELECT
    *,
    LPAD(
      REGEXP_REPLACE(CAST(cod_caen_mfinante AS STRING), '[^0-9]', ''),
      4,
      '0'
    ) AS cod_caen_mfinante_4
  FROM company_ro.silver.fact_financiar_anual
)

SELECT
  f.an,
  f.cui,

  d.denumire,
  d.cod_inmatriculare,
  d.judet,
  d.localitate,
  d.adresa,
  d.cod_stare_firma,
  d.stare_firma,
  d.is_activa,
  d.forma_juridica,
  d.an_infiintare,

  f.cod_caen_mfinante_4 AS cod_caen_mfinante,

  COALESCE(c.cod_caen, f.cod_caen_mfinante_4) AS cod_caen,
  c.denumire AS denumire_caen,
  c.grupa AS grupa_caen,
  c.clasa AS clasa_caen,

  CONCAT(
    f.cod_caen_mfinante_4,
    ' - ',
    COALESCE(c.denumire, 'Unknown CAEN')
  ) AS caen_display_name,

  f.cifra_afaceri,
  f.venituri_totale,
  f.cheltuieli_totale,
  f.profit_brut,
  f.pierdere_bruta,
  f.profit_net,
  f.pierdere_neta,
  f.nr_mediu_salariati,

  f.datorii,
  f.capitaluri_proprii,
  f.capital_social,
  f.active_imobilizate,
  f.active_circulante,
  f.stocuri,
  f.creante,
  f.casa_si_conturi,

  f._ingested_at,
  f._source_file

FROM fact_normalized f

LEFT JOIN company_ro.silver.dim_firma d
  ON f.cui = d.cui

LEFT JOIN company_ro.silver.dim_caen c
  ON f.cod_caen_mfinante_4 = c.cod_caen
""")

display(company_financial_summary.limit(50))

print("Rows:", company_financial_summary.count())
print("Columns:", company_financial_summary.columns)

In [0]:
display(company_financial_summary.groupBy("an").agg(
    F.count("*").alias("rows"),
    F.countDistinct("cui").alias("companies"),
    F.count("denumire").alias("rows_with_company_name"),
    F.count("cod_caen_mfinante").alias("rows_with_caen_code"),
    F.count("denumire_caen").alias("rows_with_caen_name"),
    F.count("cifra_afaceri").alias("rows_with_cifra_afaceri"),
    F.count("profit_net").alias("rows_with_profit_net"),
    F.count("nr_mediu_salariati").alias("rows_with_salariati")
).orderBy("an"))

In [0]:
display(company_financial_summary.groupBy(
    "cod_caen",
    "denumire_caen",
    "grupa_caen",
    "clasa_caen",
    "caen_display_name"
).agg(
    F.count("*").alias("rows")
).orderBy(F.col("rows").desc()).limit(100))

In [0]:
(
    company_financial_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("company_ro.gold.company_financial_summary")
)

In [0]:
display(spark.sql("""
SELECT
  an,
  COUNT(*) AS rows,
  COUNT(DISTINCT cui) AS companies,
  SUM(cifra_afaceri) AS total_cifra_afaceri,
  SUM(profit_net) AS total_profit_net,
  SUM(pierdere_neta) AS total_pierdere_neta,
  SUM(nr_mediu_salariati) AS total_salariati
FROM company_ro.gold.company_financial_summary
GROUP BY an
ORDER BY an
"""))

In [0]:
display(spark.sql("""
SELECT
  cod_caen,
  denumire_caen,
  grupa_caen,
  caen_display_name,
  COUNT(*) AS rows
FROM company_ro.gold.company_financial_summary
GROUP BY
  cod_caen,
  denumire_caen,
  grupa_caen,
  caen_display_name
ORDER BY rows DESC
LIMIT 100
"""))

In [0]:
display(spark.sql("""
WITH normalized AS (
  SELECT
    *,
    UPPER(
      TRANSLATE(
        judet,
        'ĂÂÎȘŞȚŢăâîșşțţ',
        'AAISSTTaaisstt'
      )
    ) AS judet_norm
  FROM company_ro.gold.company_financial_summary
)

SELECT
  an,
  cui,
  denumire,
  judet,
  localitate,
  cod_caen,
  denumire_caen,
  grupa_caen,
  clasa_caen,
  caen_display_name,
  cifra_afaceri,
  profit_net,
  pierdere_neta,
  nr_mediu_salariati
FROM normalized
WHERE judet_norm = 'IASI'
  AND (
    cod_caen LIKE '62%'
    OR cod_caen LIKE '63%'
  )
ORDER BY an DESC, cifra_afaceri DESC
LIMIT 200
"""))

In [0]:
display(spark.sql("""
SELECT
  an,
  grupa_caen,
  COUNT(DISTINCT cui) AS companies,
  SUM(cifra_afaceri) AS total_cifra_afaceri,
  SUM(profit_net) AS total_profit_net,
  SUM(nr_mediu_salariati) AS total_salariati
FROM company_ro.gold.company_financial_summary
WHERE grupa_caen IS NOT NULL
GROUP BY
  an,
  grupa_caen
ORDER BY an DESC, total_cifra_afaceri DESC
LIMIT 100
"""))

In [0]:
display(spark.sql("""
SELECT
  COUNT(*) AS rows,
  COUNT(cod_caen_mfinante) AS rows_with_cod_caen_mfinante,
  COUNT(cod_caen) AS rows_with_cod_caen,
  COUNT(denumire_caen) AS rows_with_denumire_caen,
  COUNT(grupa_caen) AS rows_with_grupa_caen,
  COUNT(clasa_caen) AS rows_with_clasa_caen,
  COUNT(caen_display_name) AS rows_with_caen_display_name
FROM company_ro.gold.company_financial_summary
"""))

In [0]:
display(spark.sql("""
SELECT
  cod_caen,
  denumire_caen,
  grupa_caen,
  clasa_caen,
  caen_display_name,
  COUNT(*) AS rows
FROM company_ro.gold.company_financial_summary
GROUP BY
  cod_caen,
  denumire_caen,
  grupa_caen,
  clasa_caen,
  caen_display_name
ORDER BY rows DESC
LIMIT 100
"""))

In [0]:
display(spark.sql("""
SELECT
  an,
  cui,
  denumire,
  judet,
  localitate,
  cod_caen,
  denumire_caen,
  grupa_caen,
  clasa_caen,
  caen_display_name,
  cifra_afaceri,
  profit_net,
  nr_mediu_salariati
FROM company_ro.gold.company_financial_summary
WHERE cod_caen LIKE '62%'
   OR cod_caen LIKE '63%'
ORDER BY an DESC, cifra_afaceri DESC
LIMIT 100
"""))

In [0]:
sample_cui = spark.sql("""
SELECT
  cui,
  COUNT(*) AS years
FROM company_ro.silver.fact_financiar_anual
GROUP BY cui
HAVING COUNT(*) >= 5
ORDER BY years DESC
LIMIT 1
""").collect()[0]["cui"]

print("Using CUI:", sample_cui)

display(spark.sql(f"""
SELECT
  f.an,
  f.cui,
  d.denumire,
  d.judet,
  d.localitate,
  d.stare_firma,
  f.cod_caen_mfinante,
  f.cifra_afaceri,
  f.profit_net,
  f.pierdere_neta,
  f.nr_mediu_salariati,
  f.datorii,
  f.capitaluri_proprii,
  f.venituri_totale,
  f.cheltuieli_totale
FROM company_ro.silver.fact_financiar_anual f
LEFT JOIN company_ro.silver.dim_firma d
  ON f.cui = d.cui
WHERE f.cui = '{sample_cui}'
ORDER BY f.an
"""))


## Data Quality Checks - Gold Layer

Validation tests to ensure data quality and join coverage in the denormalized gold table.

In [0]:
# Data Quality Check 1: Row count reconciliation with silver fact table
print("=== DQ Check 1: Row Count Reconciliation ===")

silver_count = spark.sql("""
SELECT COUNT(*) AS row_count
FROM company_ro.silver.fact_financiar_anual
""").collect()[0]["row_count"]

gold_count = spark.sql("""
SELECT COUNT(*) AS row_count
FROM company_ro.gold.company_financial_summary
""").collect()[0]["row_count"]

print(f"Silver fact table rows: {silver_count:,}")
print(f"Gold summary table rows: {gold_count:,}")

if silver_count == gold_count:
    print("PASS: Row counts match perfectly")
else:
    diff = silver_count - gold_count
    diff_pct = abs(diff / silver_count * 100) if silver_count > 0 else 0
    print(f"MISMATCH: Difference of {diff:,} rows ({diff_pct:.2f}%)")

In [0]:
# Data Quality Check 2: Join coverage analysis
print("\n=== DQ Check 2: Join Coverage Analysis ===")

join_coverage = spark.sql("""
SELECT
  COUNT(*) AS total_rows,
  
  -- Company dimension coverage
  COUNT(denumire) AS rows_with_company_name,
  ROUND(COUNT(denumire) / COUNT(*) * 100, 2) AS pct_with_company_name,
  
  COUNT(judet) AS rows_with_judet,
  ROUND(COUNT(judet) / COUNT(*) * 100, 2) AS pct_with_judet,
  
  -- CAEN dimension coverage
  COUNT(cod_caen) AS rows_with_caen_code,
  ROUND(COUNT(cod_caen) / COUNT(*) * 100, 2) AS pct_with_caen_code,
  
  COUNT(denumire_caen) AS rows_with_caen_name,
  ROUND(COUNT(denumire_caen) / COUNT(*) * 100, 2) AS pct_with_caen_name,
  
  COUNT(grupa_caen) AS rows_with_grupa_caen,
  ROUND(COUNT(grupa_caen) / COUNT(*) * 100, 2) AS pct_with_grupa_caen,
  
  -- Financial data completeness
  COUNT(cifra_afaceri) AS rows_with_revenue,
  ROUND(COUNT(cifra_afaceri) / COUNT(*) * 100, 2) AS pct_with_revenue,
  
  COUNT(nr_mediu_salariati) AS rows_with_employees,
  ROUND(COUNT(nr_mediu_salariati) / COUNT(*) * 100, 2) AS pct_with_employees
  
FROM company_ro.gold.company_financial_summary
""")

display(join_coverage)

result = join_coverage.collect()[0]
if result["pct_with_company_name"] < 95:
    print(f"WARNING: Only {result['pct_with_company_name']}% of rows have company names")
else:
    print(f"GOOD: {result['pct_with_company_name']}% of rows have company names")
    
if result["pct_with_caen_name"] < 90:
    print(f"WARNING: Only {result['pct_with_caen_name']}% of rows have CAEN descriptions")
else:
    print(f"GOOD: {result['pct_with_caen_name']}% of rows have CAEN descriptions")

In [0]:
# Data Quality Check 3: Enrichment quality trends by year
print("\n=== DQ Check 3: Enrichment Quality Trends by Year ===")

enrichment_by_year = spark.sql("""
SELECT
  an,
  COUNT(*) AS total_rows,
  COUNT(denumire) AS with_company_name,
  ROUND(COUNT(denumire) / COUNT(*) * 100, 2) AS pct_company_name,
  
  COUNT(denumire_caen) AS with_caen_name,
  ROUND(COUNT(denumire_caen) / COUNT(*) * 100, 2) AS pct_caen_name,
  
  COUNT(grupa_caen) AS with_grupa_caen,
  ROUND(COUNT(grupa_caen) / COUNT(*) * 100, 2) AS pct_grupa_caen
  
FROM company_ro.gold.company_financial_summary
GROUP BY an
ORDER BY an DESC
""")

display(enrichment_by_year)
print("Year-by-year enrichment analysis complete")

In [0]:
# Data Quality Check 4: Null analysis on key financial metrics
print("\n=== DQ Check 4: Null Analysis on Key Financial Metrics ===")

null_analysis = spark.sql("""
SELECT
  'cifra_afaceri' AS metric,
  COUNT(*) AS total_rows,
  COUNT(cifra_afaceri) AS non_null_rows,
  ROUND(COUNT(cifra_afaceri) / COUNT(*) * 100, 2) AS pct_populated
FROM company_ro.gold.company_financial_summary

UNION ALL

SELECT
  'profit_net' AS metric,
  COUNT(*) AS total_rows,
  COUNT(profit_net) AS non_null_rows,
  ROUND(COUNT(profit_net) / COUNT(*) * 100, 2) AS pct_populated
FROM company_ro.gold.company_financial_summary

UNION ALL

SELECT
  'pierdere_neta' AS metric,
  COUNT(*) AS total_rows,
  COUNT(pierdere_neta) AS non_null_rows,
  ROUND(COUNT(pierdere_neta) / COUNT(*) * 100, 2) AS pct_populated
FROM company_ro.gold.company_financial_summary

UNION ALL

SELECT
  'nr_mediu_salariati' AS metric,
  COUNT(*) AS total_rows,
  COUNT(nr_mediu_salariati) AS non_null_rows,
  ROUND(COUNT(nr_mediu_salariati) / COUNT(*) * 100, 2) AS pct_populated
FROM company_ro.gold.company_financial_summary

UNION ALL

SELECT
  'datorii' AS metric,
  COUNT(*) AS total_rows,
  COUNT(datorii) AS non_null_rows,
  ROUND(COUNT(datorii) / COUNT(*) * 100, 2) AS pct_populated
FROM company_ro.gold.company_financial_summary
""")

display(null_analysis)
print("Financial metrics completeness analysis done")

In [0]:
# Data Quality Check 5: CAEN code normalization validation
print("\n=== DQ Check 5: CAEN Code Normalization Validation ===")

caen_validation = spark.sql("""
SELECT
  'Total records' AS check_type,
  COUNT(*) AS count,
  NULL AS example_codes
FROM company_ro.gold.company_financial_summary

UNION ALL

SELECT
  'Records with 4-digit CAEN codes' AS check_type,
  COUNT(*) AS count,
  NULL AS example_codes
FROM company_ro.gold.company_financial_summary
WHERE LENGTH(cod_caen_mfinante) = 4

UNION ALL

SELECT
  'Records with non-4-digit CAEN codes' AS check_type,
  COUNT(*) AS count,
  CONCAT_WS(', ', COLLECT_SET(cod_caen_mfinante)) AS example_codes
FROM (
  SELECT cod_caen_mfinante
  FROM company_ro.gold.company_financial_summary
  WHERE LENGTH(cod_caen_mfinante) != 4
    AND cod_caen_mfinante IS NOT NULL
  LIMIT 10
)
""")

display(caen_validation)

non_normalized = spark.sql("""
SELECT COUNT(*) AS cnt
FROM company_ro.gold.company_financial_summary
WHERE LENGTH(cod_caen_mfinante) != 4
  AND cod_caen_mfinante IS NOT NULL
""").collect()[0]["cnt"]

if non_normalized > 0:
    print(f"WARNING: Found {non_normalized:,} records with non-normalized CAEN codes")
else:
    print("PASS: All CAEN codes are properly normalized to 4 digits")

In [0]:
# Overall Gold Layer Health Summary
print("\n" + "="*60)
print("=== GOLD LAYER DATA QUALITY SUMMARY ===")
print("="*60)

summary = spark.sql("""
SELECT
  COUNT(*) AS total_records,
  COUNT(DISTINCT an) AS years_covered,
  MIN(an) AS earliest_year,
  MAX(an) AS latest_year,
  COUNT(DISTINCT cui) AS unique_companies,
  
  ROUND(COUNT(denumire) / COUNT(*) * 100, 2) AS pct_with_company_info,
  ROUND(COUNT(denumire_caen) / COUNT(*) * 100, 2) AS pct_with_caen_info,
  ROUND(COUNT(cifra_afaceri) / COUNT(*) * 100, 2) AS pct_with_revenue,
  
  ROUND(SUM(cifra_afaceri) / 1000000000, 2) AS total_revenue_billions,
  ROUND(SUM(profit_net) / 1000000000, 2) AS total_profit_billions,
  SUM(nr_mediu_salariati) AS total_employees
  
FROM company_ro.gold.company_financial_summary
""")

result = summary.collect()[0]

print(f"\nTotal records: {result['total_records']:,}")
print(f"Years covered: {result['years_covered']} ({result['earliest_year']}-{result['latest_year']})")
print(f"Unique companies: {result['unique_companies']:,}")
print(f"\nData Completeness:")
print(f"  - Company info: {result['pct_with_company_info']}%")
print(f"  - CAEN info: {result['pct_with_caen_info']}%")
print(f"  - Revenue data: {result['pct_with_revenue']}%")
print(f"\nAggregate Metrics:")
print(f"  - Total revenue: {result['total_revenue_billions']:.2f}B RON")
print(f"  - Total profit: {result['total_profit_billions']:.2f}B RON")
print(f"  - Total employees: {result['total_employees']:,}")
print("\n" + "="*60)
print("All gold layer data quality checks completed.")
print("="*60)